# Complete Guide to Recursion: Part 3

## Advanced Recursion Concepts

In this part, we'll cover:
1. Memoization (Caching results)
2. Backtracking
3. Tree Recursion Patterns
4. Common Pitfalls and Debugging

---

## Task 1: Memoization - Making Recursion Efficient

### The Problem: Repeated Calculations

Remember Fibonacci from Part 2? It was VERY inefficient because it calculated the same values multiple times.

In [ ]:
# Inefficient Fibonacci - calculates same values repeatedly
call_count = 0

def fib_slow(n):
    global call_count
    call_count += 1
    
    if n <= 1:
        return n
    return fib_slow(n - 1) + fib_slow(n - 2)

# Test it
call_count = 0
result = fib_slow(10)
print(f"fib(10) = {result}")
print(f"Function called {call_count} times!")

# Try with larger number
call_count = 0
result = fib_slow(20)
print(f"\nfib(20) = {result}")
print(f"Function called {call_count} times!")

**Why so many calls?**

```
fib(5) calls:
  fib(4) and fib(3)
  
fib(4) calls:
  fib(3) and fib(2)  ← fib(3) calculated AGAIN!
  
fib(3) is calculated TWICE just for fib(5)!
For fib(20), fib(3) is calculated THOUSANDS of times!
```

**Solution: Memoization** - Store results so we don't recalculate them!

### Method 1: Using a Dictionary (Manual Memoization)

In [ ]:
def fib_memo(n, memo=None):
    # Initialize memo dictionary on first call
    if memo is None:
        memo = {}
    
    print(f"Calculating fib({n})")
    
    # Check if we already calculated this
    if n in memo:
        print(f"  → Found in cache! fib({n}) = {memo[n]}")
        return memo[n]
    
    # Base cases
    if n <= 1:
        print(f"  → Base case: fib({n}) = {n}")
        return n
    
    # Calculate and store in memo
    print(f"  → Computing fib({n})...")
    result = fib_memo(n - 1, memo) + fib_memo(n - 2, memo)
    
    # Store result before returning
    memo[n] = result
    print(f"  → Stored fib({n}) = {result} in cache")
    
    return result

# Test it
print("Computing fib(6) with memoization:\n")
result = fib_memo(6)
print(f"\nFinal result: {result}")

**What's happening:**

1. Before calculating, check if result is in `memo` dictionary
2. If found, return it immediately (no recursion needed!)
3. If not found, calculate it and store in `memo`
4. Next time we need it, it's already there!

**Execution for fib_memo(5):**
```
fib(5) → not in memo, calculate
  fib(4) → not in memo, calculate
    fib(3) → not in memo, calculate
      fib(2) → not in memo, calculate
        fib(1) → base case, return 1
        fib(0) → base case, return 0
      fib(2) = 1, STORE in memo
      fib(1) → base case, return 1
    fib(3) = 2, STORE in memo
    fib(2) → FOUND in memo! return 1 (no recursion!)
  fib(4) = 3, STORE in memo
  fib(3) → FOUND in memo! return 2 (no recursion!)
fib(5) = 5, STORE in memo
```

### Comparison: With vs Without Memoization

In [ ]:
import time

# Without memoization
def fib_slow(n):
    if n <= 1:
        return n
    return fib_slow(n - 1) + fib_slow(n - 2)

# With memoization
def fib_fast(n, memo=None):
    if memo is None:
        memo = {}
    if n in memo:
        return memo[n]
    if n <= 1:
        return n
    memo[n] = fib_fast(n - 1, memo) + fib_fast(n - 2, memo)
    return memo[n]

# Compare performance
n = 30

start = time.time()
result1 = fib_slow(n)
time1 = time.time() - start

start = time.time()
result2 = fib_fast(n)
time2 = time.time() - start

print(f"fib({n}) = {result1}")
print(f"\nWithout memoization: {time1:.4f} seconds")
print(f"With memoization: {time2:.4f} seconds")
print(f"\nSpeedup: {time1/time2:.0f}x faster!")

### Method 2: Using Python's @lru_cache Decorator

In [ ]:
from functools import lru_cache

# Automatic memoization with decorator!
@lru_cache(maxsize=None)  # maxsize=None means unlimited cache
def fib_cached(n):
    if n <= 1:
        return n
    return fib_cached(n - 1) + fib_cached(n - 2)

# Test it
print("Using @lru_cache decorator:")
for i in range(10):
    print(f"fib({i}) = {fib_cached(i)}")

# Check cache statistics
print(f"\nCache info: {fib_cached.cache_info()}")

**@lru_cache explained:**

- **LRU** = Least Recently Used
- Automatically caches function results
- No need to manually manage memo dictionary
- `cache_info()` shows hits (found in cache) vs misses (had to calculate)

**When to use memoization:**
1. Function is called multiple times with same arguments
2. Function is pure (same input always gives same output)
3. Calculation is expensive
4. You have overlapping subproblems

### Example: Memoization with Multiple Parameters

In [ ]:
# Calculate binomial coefficient C(n, k) = n! / (k! * (n-k)!)
# Also known as "n choose k"

@lru_cache(maxsize=None)
def binomial(n, k):
    print(f"Calculating C({n}, {k})")
    
    # Base cases
    if k == 0 or k == n:
        return 1
    
    # Recursive formula: C(n,k) = C(n-1,k-1) + C(n-1,k)
    return binomial(n - 1, k - 1) + binomial(n - 1, k)

# Test it
result = binomial(5, 2)
print(f"\nC(5, 2) = {result}")
print(f"Cache info: {binomial.cache_info()}")

# Calculate another - notice fewer calculations due to cache
print("\n" + "="*50)
result = binomial(6, 3)
print(f"\nC(6, 3) = {result}")
print(f"Cache info: {binomial.cache_info()}")

### Practice: Memoization

In [ ]:
# Problem 1: Implement memoized version of counting ways to climb stairs
# You can climb 1 or 2 steps at a time. How many ways to reach step n?

@lru_cache(maxsize=None)
def climb_stairs(n):
    # Your code here
    pass

# Test: climb_stairs(5) should return 8
# Ways: 1+1+1+1+1, 1+1+1+2, 1+1+2+1, 1+2+1+1, 2+1+1+1, 1+2+2, 2+1+2, 2+2+1

In [ ]:
# Problem 2: Grid paths - count paths from (0,0) to (m,n) moving only right or down

@lru_cache(maxsize=None)
def grid_paths(m, n):
    # Your code here
    pass

# Test: grid_paths(3, 3) should return 6

### Key Takeaways - Task 1:

1. **Memoization** = Caching results to avoid recalculation
2. **Manual memoization**: Use a dictionary to store results
3. **@lru_cache**: Python's built-in decorator for automatic memoization
4. **Huge performance gains**: Can turn exponential time into linear time!
5. **Trade-off**: Uses more memory to save computation time

---

## Task 2: Backtracking - Exploring All Possibilities

### What is Backtracking?

**Backtracking** is a technique where you:
1. Make a choice
2. Explore that choice recursively
3. **Undo the choice** (backtrack) and try another option

Think of it like exploring a maze:
- Try a path
- If it's a dead end, go back and try another path
- Keep trying until you find all solutions

### Example 1: Generate All Binary Strings of Length n

In [ ]:
def generate_binary(n):
    """
    Generate all binary strings of length n
    Example: n=3 → ['000', '001', '010', '011', '100', '101', '110', '111']
    """
    result = []
    
    def backtrack(current):
        print(f"Current string: '{current}'" + (" ← Complete!" if len(current) == n else ""))
        
        # Base Case: reached desired length
        if len(current) == n:
            result.append(current)
            return
        
        # Choice 1: Add '0'
        print(f"  → Trying '0'...")
        backtrack(current + '0')
        
        # Choice 2: Add '1'
        print(f"  → Trying '1'...")
        backtrack(current + '1')
        
        print(f"  ← Backtracking from '{current}'" )
    
    backtrack('')
    return result

# Test it
print("Generating binary strings of length 2:\n")
result = generate_binary(2)
print(f"\nResult: {result}")

**Execution Tree for n=2:**

```
                    ''
                   /  \
                 '0'  '1'
                /  \   /  \
              '00' '01' '10' '11'  ← All complete!
```

**Key Pattern:**
1. Make a choice (add '0' or '1')
2. Recurse with that choice
3. When recursion returns, we've "backtracked" to try the next choice

### Example 2: Generate All Permutations

In [ ]:
def permutations(arr):
    """
    Generate all permutations of an array
    Example: [1, 2, 3] → [[1,2,3], [1,3,2], [2,1,3], [2,3,1], [3,1,2], [3,2,1]]
    """
    result = []
    
    def backtrack(current, remaining):
        print(f"Current: {current}, Remaining: {remaining}")
        
        # Base Case: no more elements to add
        if not remaining:
            result.append(current[:])  # Make a copy!
            print(f"  → Found permutation: {current}")
            return
        
        # Try each remaining element
        for i in range(len(remaining)):
            # Choose: pick element at index i
            chosen = remaining[i]
            print(f"  → Choosing {chosen}")
            
            # Explore: add to current, remove from remaining
            current.append(chosen)
            new_remaining = remaining[:i] + remaining[i+1:]
            backtrack(current, new_remaining)
            
            # Unchoose: backtrack (remove from current)
            current.pop()
            print(f"  ← Backtracked, removed {chosen}")
    
    backtrack([], arr)
    return result

# Test it
print("Generating permutations of [1, 2, 3]:\n")
result = permutations([1, 2, 3])
print(f"\nAll permutations: {result}")

**The Backtracking Pattern:**

```python
def backtrack(current_state):
    if is_solution(current_state):
        save_solution(current_state)
        return
    
    for choice in available_choices:
        # 1. MAKE the choice
        make_choice(choice)
        
        # 2. EXPLORE with that choice
        backtrack(new_state)
        
        # 3. UNDO the choice (backtrack)
        undo_choice(choice)
```

**Critical:** The "undo" step is what makes it backtracking!

### Example 3: Subsets (Power Set)

In [ ]:
def subsets(nums):
    """
    Generate all subsets of an array
    Example: [1, 2] → [[], [1], [2], [1, 2]]
    """
    result = []
    
    def backtrack(start, current):
        # Every state is a valid subset!
        result.append(current[:])
        print(f"Added subset: {current}")
        
        # Try adding each remaining element
        for i in range(start, len(nums)):
            # Choose: add nums[i]
            current.append(nums[i])
            print(f"  → Added {nums[i]}, current: {current}")
            
            # Explore: continue from next index
            backtrack(i + 1, current)
            
            # Unchoose: remove nums[i]
            removed = current.pop()
            print(f"  ← Removed {removed}, current: {current}")
    
    backtrack(0, [])
    return result

# Test it
print("Generating subsets of [1, 2, 3]:\n")
result = subsets([1, 2, 3])
print(f"\nAll subsets ({len(result)} total): {result}")

**Execution Tree for [1, 2]:**

```
                    []
                   /  \
                [1]    [2]
                /
            [1,2]

Subsets found: [], [1], [1,2], [2]
```

**Note:** We use `start` parameter to avoid duplicates (don't go backwards)

### Practice: Backtracking

In [ ]:
# Problem 1: Generate all combinations of k numbers from 1 to n
# Example: combinations(4, 2) → [[1,2], [1,3], [1,4], [2,3], [2,4], [3,4]]

def combinations(n, k):
    # Your code here
    pass

# Test: combinations(4, 2)

In [ ]:
# Problem 2: Letter combinations of a phone number
# Example: '23' → ['ad', 'ae', 'af', 'bd', 'be', 'bf', 'cd', 'ce', 'cf']
# Phone mapping: 2→abc, 3→def, 4→ghi, 5→jkl, 6→mno, 7→pqrs, 8→tuv, 9→wxyz

def letter_combinations(digits):
    # Your code here
    pass

# Test: letter_combinations('23')

### Key Takeaways - Task 2:

1. **Backtracking** = Make choice → Explore → Undo choice
2. **Three steps**: Choose, Explore, Unchoose
3. **Explores all possibilities** systematically
4. **Common uses**: Permutations, combinations, subsets, puzzles
5. **Key insight**: The "undo" step lets you try different paths

---

**Next: Task 3 - Tree Recursion Patterns**

## Task 3: Tree Recursion Patterns

### Understanding Tree Recursion

Tree recursion is when you make **multiple recursive calls** on tree-like data structures.

**Key Concepts:**
1. Each node can have left and right children
2. Process current node
3. Recursively process left subtree
4. Recursively process right subtree

### Pattern 1: Tree Traversals

In [ ]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

# Preorder: Root → Left → Right
def preorder(node):
    if not node:
        return
    print(node.val, end=" ")  # Process root FIRST
    preorder(node.left)        # Then left
    preorder(node.right)       # Then right

# Inorder: Left → Root → Right
def inorder(node):
    if not node:
        return
    inorder(node.left)         # Process left FIRST
    print(node.val, end=" ")  # Then root
    inorder(node.right)        # Then right

# Postorder: Left → Right → Root
def postorder(node):
    if not node:
        return
    postorder(node.left)       # Process left FIRST
    postorder(node.right)      # Then right
    print(node.val, end=" ")  # Then root LAST

# Create a tree:
#       1
#      / \
#     2   3
#    / \
#   4   5
root = TreeNode(1)
root.left = TreeNode(2)
root.right = TreeNode(3)
root.left.left = TreeNode(4)
root.left.right = TreeNode(5)

print("Preorder:  ", end="")
preorder(root)
print("\nInorder:   ", end="")
inorder(root)
print("\nPostorder: ", end="")
postorder(root)
print()

**When to use each traversal:**

- **Preorder**: Copy tree, prefix expression
- **Inorder**: Get sorted values from BST
- **Postorder**: Delete tree, postfix expression

### Pattern 2: Calculating Tree Properties

In [ ]:
# Height of tree
def height(node):
    if not node:
        return 0
    
    left_height = height(node.left)
    right_height = height(node.right)
    
    return max(left_height, right_height) + 1

# Count nodes
def count_nodes(node):
    if not node:
        return 0
    
    return 1 + count_nodes(node.left) + count_nodes(node.right)

# Sum of all values
def tree_sum(node):
    if not node:
        return 0
    
    return node.val + tree_sum(node.left) + tree_sum(node.right)

# Test with our tree
print(f"Height: {height(root)}")
print(f"Node count: {count_nodes(root)}")
print(f"Sum: {tree_sum(root)}")

### Pattern 3: Returning Multiple Values (Your Tree Balance Problem!)

In [ ]:
def is_balanced_detailed(root):
    """
    Check if tree is balanced AND return height
    Returns: (height, is_balanced)
    """
    def helper(node, depth=0):
        indent = "  " * depth
        print(f"{indent}Visiting: {node.val if node else 'None'}")
        
        # Base Case
        if not node:
            print(f"{indent}  → Returning (0, True)")
            return (0, True)
        
        # Get info from left subtree
        print(f"{indent}  Going LEFT...")
        left_height, left_balanced = helper(node.left, depth + 1)
        print(f"{indent}  ← Left returned: height={left_height}, balanced={left_balanced}")
        
        # Get info from right subtree
        print(f"{indent}  Going RIGHT...")
        right_height, right_balanced = helper(node.right, depth + 1)
        print(f"{indent}  ← Right returned: height={right_height}, balanced={right_balanced}")
        
        # Calculate current node's info
        height = max(left_height, right_height) + 1
        is_balanced = (left_balanced and 
                      right_balanced and 
                      abs(left_height - right_height) <= 1)
        
        print(f"{indent}  → Node {node.val}: height={height}, balanced={is_balanced}")
        return (height, is_balanced)
    
    return helper(root)

# Test with balanced tree
print("Testing balanced tree:\n")
balanced_tree = TreeNode(1, TreeNode(2), TreeNode(3))
h, b = is_balanced_detailed(balanced_tree)
print(f"\nResult: Height={h}, Balanced={b}")

print("\n" + "="*60)

# Test with unbalanced tree
print("\nTesting unbalanced tree:\n")
unbalanced_tree = TreeNode(1)
unbalanced_tree.left = TreeNode(2)
unbalanced_tree.left.left = TreeNode(3)
h, b = is_balanced_detailed(unbalanced_tree)
print(f"\nResult: Height={h}, Balanced={b}")

**This is EXACTLY your tree problem!**

1. **Base case**: Empty node returns (0, True)
2. **Recursive calls**: Get info from BOTH children
3. **Tuple unpacking**: Extract height and balance status
4. **Combine results**: Use children's info to calculate current node's info
5. **Return tuple**: Pass info back up to parent

### Practice: Tree Recursion

In [ ]:
# Problem 1: Find maximum value in tree
def find_max(node):
    # Your code here
    pass

# Test: find_max(root) should return 5

In [ ]:
# Problem 2: Count leaf nodes (nodes with no children)
def count_leaves(node):
    # Your code here
    pass

# Test: count_leaves(root) should return 3

### Key Takeaways - Task 3:

1. **Tree recursion** = Multiple recursive calls (left and right)
2. **Base case**: Always handle empty node (None)
3. **Three steps**: Process current, recurse left, recurse right (order matters!)
4. **Returning tuples**: Pass multiple pieces of information up the tree
5. **Common patterns**: Traversals, properties, comparisons, paths

---

**End of Part 3**

You now have a complete understanding of recursion from basics to advanced tree patterns!